In [1]:
from collections import defaultdict

In [2]:
from aana.api.editor import (
    deployment_output_to_outputs,
    get_all_deployments,
    get_class_names_from_file,
    get_deployment_methods,
    get_function_details,
    get_pydantic_models,
)
from aana.configs.deployments import deployments as all_deployments
from importlib import resources
import os

deployment_class_to_name = {
    deployment.name: deployment_name
    for deployment_name, deployment in all_deployments.items()
}

pydantic_models = get_pydantic_models()
pydantic_model_names = [m.name for m in pydantic_models]

# outputs = {}
# output_dicts = get_deployments_typed_dicts()
# for output in output_dicts:
#     # print(output.name)
#     # print(deployment_output_to_outputs(output))
#     outputs[output.name] = deployment_output_to_outputs(output)
# walk all the files in the aana and get all the typed dicts
typed_dicts = []
for root, dirs, files in os.walk(resources.path("aana", "")):
    for file in files:
        # Check if the file ends with .py
        if file.endswith(".py"):
            path = os.path.join(root, file)
            # print(path)
            classes = get_class_names_from_file(path)
            # keep only TypedDict
            typed_dicts.extend(
                [
                    c
                    for c in classes
                    if "TypedDict" in [c.id for c in c.bases if hasattr(c, "id")]
                ]
            )
outputs = {}
for output in typed_dicts:
    # print(output.name)
    # print(deployment_output_to_outputs(output))
    outputs[output.name] = deployment_output_to_outputs(output)

deployments = {}
for deployment in get_all_deployments():
    deployments[deployment_class_to_name[deployment.name]] = {}
    for method in get_deployment_methods(deployment):
        deployments[deployment_class_to_name[deployment.name]][method["name"]] = {
            "inputs": method["args"],
            "is_generator": method["is_generator"],
            # "return_type": method["return_type"],
            "outputs": outputs.get(method["return_type"], []),
        }

available_functions = [
    "aana.utils.video.download_video",
    "aana.utils.video.extract_frames_decord",
    "aana.utils.video.generate_frames_decord",
]

functions = {}
for func in available_functions:
    functions[func] = get_function_details(func, outputs)

/root/.cache/pypoetry/virtualenvs/aana-vIr3-B0u-py3.10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2024-03-01 11:28:17,831	INFO util.py:159 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.
2024-03-01 11:28:19,448	INFO util.py:159 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


In [3]:
# pipeline = [
#     {
#         "name": "prompt",
#         "type": "input",
#         "inputs": [],
#         "outputs": [
#             {
#                 "name": "prompt",
#                 "key": "prompt",
#                 "path": "prompt",
#                 "data_model": "Prompt",
#             }
#         ],
#     },
#     {
#         "name": "vllm_stream_llama2_7b_chat",
#         "type": "ray_deployment",
#         "deployment_name": "vllm_deployment_llama2_7b_chat",
#         "data_type": "generator",
#         "generator_path": "prompt",
#         "method": "generate_stream",
#         "inputs": [
#             {"name": "prompt", "key": "prompt", "path": "prompt"},
#             {
#                 "name": "sampling_params",
#                 "key": "sampling_params",
#                 "path": "sampling_params",
#             },
#         ],
#         "outputs": [
#             {
#                 "name": "vllm_llama2_7b_chat_output_stream",
#                 "key": "text",
#                 "path": "vllm_llama2_7b_chat_output_stream",
#             }
#         ],
#     },
# ]

In [4]:
pipeline = [
    {
        "name": "video",
        "type": "input",
        "inputs": [],
        "outputs": [
            {
                "name": "video",
                "key": "video",
                "path": "video.video_input",
                "data_model": "VideoInput",
            }
        ],
    },
    {
        "name": "video_params",
        "type": "input",
        "inputs": [],
        "outputs": [
            {
                "name": "video_params",
                "key": "video_params",
                "path": "video_batch.params",
                "data_model": "VideoParams",
            }
        ],
    },
    {
        "name": "download_video",
        "type": "ray_task",
        "function": "aana.utils.video.download_video",
        "dict_output": False,
        "inputs": [
            {
                "name": "video",
                "key": "video_input",
                "path": "video.video_input",
            },
        ],
        "outputs": [
            {
                "name": "video_object",
                "key": "output",
                "path": "video.video",
            },
        ],
    },
    {
        "name": "generate_frames_for_video",
        "type": "ray_task",
        "function": "aana.utils.video.generate_frames_decord",
        "data_type": "generator",
        "generator_path": "video",
        "kwargs": {
            "batch_size": 4,
        },
        "inputs": [
            {
                "name": "video_object",
                "key": "video",
                "path": "video.video",
            },
            {"name": "video_params", "key": "params", "path": "video_batch.params"},
        ],
        "outputs": [
            {
                "name": "video_frames",
                "key": "frames",
                "path": "video.frames.[*].image",
            },
            {
                "name": "video_frame_ids",
                "key": "frame_ids",
                "path": "video.frames.[*].id",
            },
            {
                "name": "video_timestamps",
                "key": "timestamps",
                "path": "video.timestamps",
            },
            {
                "name": "video_duration",
                "key": "duration",
                "path": "video.duration",
            },
        ],
    },
    {
        "name": "hf_blip2_opt_2_7b_video",
        "type": "ray_deployment",
        "deployment_name": "hf_blip2_deployment_opt_2_7b",
        "method": "generate_batch",
        "flatten_by": "video.frames.[*]",
        "inputs": [
            {
                "name": "video_frames",
                "key": "images",
                "path": "video.frames.[*].image",
                "partial": True,
            }
        ],
        "outputs": [
            {
                "name": "video_captions_hf_blip2_opt_2_7b",
                "key": "captions",
                "path": "video.frames.[*].caption_hf_blip2_opt_2_7b",
                "data_model": "CaptionsList",
            }
        ],
    },
]

In [5]:
endpoints = [
    {
        "name": "blip2_video_generate",
        "path": "/video/generate_captions",
        "summary": "Generate captions for videos using BLIP2 OPT-2.7B",
        "outputs": [
            {"name": "captions", "output": "video_captions_hf_blip2_opt_2_7b"},
            {"name": "timestamps", "output": "video_timestamps"},
            {"name": "caption_ids", "output": "caption_ids"},
        ],
    },
]

In [6]:
# all_outputs = {}
# for node in pipeline:
#     for input in node["outputs"]:
#         all_outputs[input["name"]] = len(all_outputs) + 1

# all_outputs

output_to_node_id = {}
for i, node in enumerate(pipeline):
    for output_index, output in enumerate(node["outputs"]):
        output_to_node_id[output["name"]] = {"node_id": i + 1, "slot": output_index}

In [7]:
last_link_id = 0
all_links = {
    "output": defaultdict(
        list
    ),  # output name to list of link ids, because multiple nodes can use the same output as input
    "input": {},  # node name + input name to link id, because connection comes from only one output
}
links = []
for node_index, node in enumerate(pipeline):
    for input_index, input in enumerate(node["inputs"]):
        all_links["input"][f"{node['name']}:{input['name']}"] = last_link_id + 1
        all_links["output"][input["name"]].append(last_link_id + 1)

        links.append(
            [
                last_link_id + 1,  # link id
                output_to_node_id[input["name"]]["node_id"],  # origin node id
                output_to_node_id[input["name"]]["slot"],  # origin slot
                node_index + 1,  # target node id
                input_index,  # target slot
                "string",  # type
            ]
        )

        last_link_id += 1

In [8]:
all_links

{'output': defaultdict(list,
             {'video': [1],
              'video_object': [2],
              'video_params': [3],
              'video_frames': [4]}),
 'input': {'download_video:video': 1,
  'generate_frames_for_video:video_object': 2,
  'generate_frames_for_video:video_params': 3,
  'hf_blip2_opt_2_7b_video:video_frames': 4}}

In [9]:
links

[[1, 1, 0, 3, 0, 'string'],
 [2, 3, 0, 4, 0, 'string'],
 [3, 2, 0, 4, 1, 'string'],
 [4, 4, 0, 5, 0, 'string']]

In [10]:
# "properties": {
#     "name": "function",
#     "function_name": "aana.utils.video.download_video",
#     "is_generator": false,
#     "dict_output": false
# },
# "widgets_values": [
#     "aana.utils.video.download_video"
# ]

In [17]:
editor_nodes = []
last_node_id = len(pipeline)
for i, node in enumerate(pipeline):
    # last_node_id += 1
    node_id = i + 1
    editor_node = {}
    editor_node["id"] = node_id
    editor_node["pos"] = [node_id * 300, node_id * 300]
    # editor_node["size"] = {"0": 400, "1": 200}
    editor_node["flags"] = {}
    editor_node["order"] = node_id
    editor_node["mode"] = 0
    editor_node["title"] = node["name"]

    # if node["name"] == "generate_frames_for_video":
    # break

    if node["type"] == "ray_task" or node["type"] == "function":
        # break

        function_name = node["function"]
        function_def = functions[function_name]

        if function_def["is_generator"] != (node.get("data_type") == "generator"):
            raise ValueError(
                f"Function {function_name} is a generator but the pipeline node is not or vice versa"
            )

        if node.get("dict_output", True) != function_def["dict_output"]:
            raise ValueError(
                f"Function {function_name} returns a dict but the pipeline node does not or vice versa"
            )

        editor_node["type"] = "pipeline/function"
        editor_node["properties"] = {
            "name": node["name"],
            "function_name": function_name,
            "is_generator": node.get("data_type") == "generator",
            "dict_output": node.get("dict_output", True),
            "is_ray_task": node["type"] == "ray_task",
            "batched": node.get("batched", False),
        }
        if node.get("data_type") == "generator":
            editor_node["properties"]["generator_path"] = node["generator_path"]
        editor_node["widgets_values"] = [node["function"], node.get("batched", False)]

    elif node["type"] == "ray_deployment":
        function_def = deployments[node["deployment_name"]][node["method"]]

        editor_node["type"] = "pipeline/deployment"

        editor_node["properties"] = {
            "name": node["name"],
            "deployment_name": node["deployment_name"],
            "method": node["method"],
            "is_generator": node.get("data_type") == "generator",
        }
        if node.get("data_type") == "generator":
            editor_node["properties"]["generator_path"] = node["generator_path"]
        editor_node["widgets_values"] = [
            node["name"],
            node["method"],
        ]
    elif node["type"] == "input":
        editor_node["type"] = "pipeline/input"
        editor_node["properties"] = {}
        editor_node["widgets_values"] = [node["name"], node["outputs"][0]["data_model"]]
    else:
        raise ValueError(f"Unknown node type {node['type']}")

    inputs = node["inputs"]
    outputs = node["outputs"]

    input_by_key = {input["key"]: input for input in inputs}
    output_by_key = {output["key"]: output for output in outputs}

    if (
        node["type"] == "ray_task"
        or node["type"] == "function"
        or node["type"] == "ray_deployment"
    ):
        input_defs = {input["name"]: input for input in function_def["inputs"]}
        output_defs = {output["name"]: output for output in function_def["outputs"]}

        for output_key in output_defs:
            path = output_by_key.get(output_key, {}).get("path", "")
            editor_node["properties"][f"path[{output_key}]"] = path
    else:
        input_defs = None
        output_defs = None

    editor_node["inputs"] = []
    for input_index, input in enumerate(inputs):
        if input_defs:
            input_def = input_defs[input["key"]]
            input_key = f"{input_def['name']} [{input_def['type']}]"
            if input_def["default"] is not None:
                input_key += f" = {input_def['default']}"
        else:
            input_key = input["name"]
        editor_node["inputs"].append(
            {
                "name": input_key,
                # "type": input.get("data_model", "Any"),
                "type": "string",
                "link": all_links["input"][f"{node['name']}:{input['name']}"],
            }
        )
    # add inputs that are in the function definition but not in the pipeline node
    if input_defs and len(inputs) < len(input_defs):
        for input_key, input_def in input_defs.items():
            if input_key not in [input["key"] for input in inputs]:
                input_name = f"{input_def['name']} [{input_def['type']}]"
                if input_def["default"] is not None:
                    input_name += f" = {input_def['default']}"
                editor_node["inputs"].append(
                    {
                        "name": input_name,
                        "type": "string",
                        "link": None,
                    }
                )

    editor_node["outputs"] = []
    for output_index, output in enumerate(outputs):
        if output_defs:
            output_def = output_defs[output["key"]]
            output_name = f"{output_def['name']} [{output_def['type']}]"
        else:
            output_name = output["name"]
        editor_node["outputs"].append(
            {
                "name": output_name,
                # "type": output.get("data_model", "Any"),
                "type": "string",
                "links": all_links["output"][output["name"]],
                "slot_index": output_index,
            }
        )

    # if one of the inputs is partial, then add property to the node to indicate that

    editor_node["properties"]["is_partial"] = False
    for input in inputs:
        if input.get("partial"):
            editor_node["properties"]["is_partial"] = True

    # if node["type"] == "ray_task" or node["type"] == "function" or node["type"] == "ray_deployment":
    #     for output_key in output_defs:
    #         path = output_by_key.get(output_key, {}).get("path", "")
    #         editor_node["properties"][f"path[{output_key}]"] = path

    if "kwargs" in node:
        kwards = node["kwargs"]
        for key, value in kwards.items():
            print(key, value)
            last_node_id += 1
            editor_kwargs_node = {
                "id": last_node_id,
                "pos": [last_node_id * 300, last_node_id * 300],
                "size": {"0": 150, "1": 100},
                "flags": {},
                "order": last_node_id,
                "mode": 0,
                "title": "Fixed Input",
                "type": "pipeline/fixed_input",
                "properties": {
                    "value": value,
                },
                "inputs": [],
                "outputs": [],
                "widgets_values": [value],
            }
            editor_nodes.append(editor_kwargs_node)

            target_slot = None
            for slot, item in enumerate(editor_node["inputs"]):
                if item["name"].startswith(f"{key} ["):
                    target_slot = slot
                    item["link"] = last_link_id + 1
                    break
            target_node_id = editor_node["id"]

            links.append(
                [
                    last_link_id + 1,  # link id
                    editor_kwargs_node["id"],  # origin node id
                    0,  # origin slot
                    target_node_id,  # target node id
                    target_slot,  # target slot
                    "string",  # type
                ]
            )
            last_link_id += 1

    editor_nodes.append(editor_node)

batch_size 4


In [18]:
for endpoint in endpoints:
    last_node_id += 1
    editor_node = {
        "id": last_node_id + 1,
        "pos": [last_node_id * 300, last_node_id * 300],
        # "size": {"0": 400, "1": 200},
        "flags": {},
        "order": last_node_id + 1,
        "mode": 0,
        "title": endpoint["name"],
        "type": "pipeline/endpoint",
        "properties": {
            "path": endpoint["path"],
            "summary": endpoint["summary"],
            "count_outputs": len(endpoint["outputs"]),
        },
        "inputs": [],
        "outputs": [],
        "widgets_values": [endpoint["path"], endpoint["summary"]],
    }
    for output_index, output in enumerate(endpoint["outputs"]):
        editor_node["properties"][f"output_{output_index}"] = output["name"]
        editor_node["properties"][f"is_streaming_{output_index}"] = output.get(
            "streaming", False
        )
        if output["output"] in output_to_node_id:
            origin_node = output_to_node_id[output["output"]]
            origin_node_obj = {
                editor_node["id"]: editor_node for editor_node in editor_nodes
            }[origin_node["node_id"]]
            links.append(
                [
                    last_link_id + 1,  # link id
                    origin_node["node_id"],  # origin node id
                    origin_node["slot"],  # origin slot
                    editor_node["id"],  # target node id
                    output_index,  # target slot
                    "string",  # type
                ]
            )
            origin_node_obj["outputs"][origin_node["slot"]]["links"].append(
                last_link_id + 1
            )
            link = (last_link_id + 1,)
            last_link_id += 1
        else:
            link = None

        editor_node["inputs"].append(
            {
                "name": f"{output['name']} (streaming)"
                if output.get("streaming")
                else output["name"],
                "type": "string",
                "link": link,
            }
        )

    editor_nodes.append(editor_node)

In [19]:
import json


with open("/workspaces/aana_sdk/aana/test_pipeline.json", "w") as f:
    f.write(
        json.dumps(
            {
                "nodes": editor_nodes,
                "links": links,
                "groups": [],
                "config": {},
                "extra": {},
                "version": 0.4,
                "last_node_id": len(editor_nodes),
                "last_link_id": last_link_id,
            },
            indent=4,
            sort_keys=True,
        )
    )